# Kalibrierung – kombiniertes Logging (MicroK + Arduino-NTC-Logger)

Ein Notebook, **zwei parallele Threads** – je ein serielles Gerät:
- **MicroK-Bridge** (SPRT-Referenz), 9600 Baud – Kanäle aus `param_combined.txt`
- **Arduino-Logger** (NTCs), 19200 Baud – Nodes/Readouts aus `param_combined.txt`

Beide schreiben je eine Datei nach `./Output/` mit `datetime`-Zeitstempel pro Zeile
→ spätere Zuordnung über die Zeitstempel.

**Ablauf:** Zellen der Reihe nach ausführen.
- Zelle 3 liest `param_combined.txt` (minimales `schlüssel: wert`-Format).
- Zelle 4 listet die seriellen Ports auf und lässt dich **interaktiv auswählen**,
  welcher Port MicroK bzw. Logger ist (Enter = Vorauswahl aus `param_combined.txt`).
- Zelle 6 startet beide Threads.

Jede Datenzeile wird sofort auf Platte geschrieben (`flush`), d. h. die Ausgabedateien
in `./Output/` können **jederzeit während des Laufs kopiert** werden.

**Stoppen:** Kernel unterbrechen (Interrupt-/Stop-Button) → beide Threads beenden
sauber und schließen die Ports.

In [ ]:
import os
import time
import threading
from datetime import datetime

import serial
import serial.tools.list_ports

In [ ]:
# ---- Konfiguration aus param_combined.txt lesen (steuert BEIDE Geräte) ----
# Minimales Format:   schluessel: wert     (Reihenfolge egal, '#' = Kommentar)
script_dir = os.path.dirname(os.path.abspath("__file__"))

cfg = {}
with open(os.path.join(script_dir, "param_combined.txt"), "r") as file:
    for line in file:
        line = line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        key, val = line.split(":", 1)
        cfg[key.strip().lower()] = val.strip()

def as_list(key):
    return [x.strip() for x in cfg[key].split(";") if x.strip()]

exp = cfg["experiment"]

# --- MicroK ---
microk_channels = as_list("microk_channels")   # Referenz; SPRT1[; SPRT2]
ref_ch      = microk_channels[0]               # Referenzkanal
sprt_chs    = microk_channels[1:]              # ein oder zwei SPRT-Kanäle
microk_hint = cfg.get("microk_port", "")       # optionale Port-Vorauswahl

# --- Logger ---
logger_hint = cfg.get("ntc_port", "")          # optionale Port-Vorauswahl
active = as_list("ntc_readout")                # z.B. NTC1; NTC2; TestSB
_g = as_list("ntc_groups")
Nr_NTCs_group = int(_g[0])                      # Sensoren pro Gruppe
Nr_MeasPoints = int(_g[1])                      # Messpunkte pro Gruppe
Logger_sensorNo = as_list("ntc_nodes")         # Node-IDs

Logger_positions = ['DateTime', 'TempADC', 'NTC1', 'NTC2', 'TestSB', 'TestN', 'GND', 'PRESSURE']
on_commands  = [f"{p} ON\r\n"  for p in Logger_positions if p in active]
off_commands = [f"{p} OFF\r\n" for p in Logger_positions if p not in active]

groups = [Logger_sensorNo[i:i + Nr_NTCs_group] for i in range(0, len(Logger_sensorNo), Nr_NTCs_group)]
commands_groupNTCs = [f"NODES {' '.join(g)} \r\n" for g in groups]

standalones = ["DateTime", "TempADC"]
standalone_cols = [s for s in standalones if s in active]
node_positions  = [a for a in active if a not in standalones]
headers = []
for group in groups:
    node_cols = [" | ".join(f"N{node.strip()}_{pos}" for pos in node_positions) for node in group]
    headers.append("SecondsElapsed; DateTimePC; " + " || ".join(standalone_cols + node_cols))

os.makedirs("Output", exist_ok=True)
run_stamp   = time.strftime("%Y%m%d-%H%M%S")
microk_file = f"Output/{exp}_{run_stamp}_microk.txt"
logger_file = f"Output/{exp}_{run_stamp}_ntc.txt"

print("Experiment :", exp)
print("MicroK     : Ref", ref_ch, "| SPRT", sprt_chs, "| Hint", microk_hint, "->", microk_file)
print("Logger     : readout", active, "| nodes", Logger_sensorNo, "| Hint", logger_hint, "->", logger_file)
print("on-cmds    :", [c.strip() for c in on_commands])
print("group-cmds :", [c.strip() for c in commands_groupNTCs])

In [ ]:
# ---- Serielle Ports zur Laufzeit auswählen ----
# Die FTDI-Namen können sich zwischen Sessions ändern; daher hier interaktiv wählen.
ports = list(serial.tools.list_ports.comports())
print("Erkannte serielle Ports:")
for idx, p in enumerate(ports):
    print(f"  [{idx}]  {p.device}   |   {p.description}   |   {p.hwid}")
print()

def pick_port(role, hint):
    # Fragt nach dem Port-Index fuer 'role'. Enter = Vorauswahl anhand 'hint' aus param.txt.
    default_idx = next((i for i, p in enumerate(ports) if hint and hint in p.device), None)
    prompt = f"Index für {role}"
    if default_idx is not None:
        prompt += f"  [Enter = {default_idx}: {ports[default_idx].device}]"
    prompt += ": "
    while True:
        choice = input(prompt).strip()
        if choice == "" and default_idx is not None:
            return ports[default_idx].device
        if choice.isdigit() and int(choice) < len(ports):
            return ports[int(choice)].device
        print("  Ungültig – bitte eine der Indexnummern oben eingeben.")

microk_port = pick_port("MicroK-Bridge (9600 Baud)", microk_hint)
logger_port = pick_port("Arduino-Logger (19200 Baud)", logger_hint)
print()
print("Gewählt  MicroK ->", microk_port)
print("Gewählt  Logger ->", logger_port)

In [ ]:
# ---- Die zwei Worker: je ein Gerät, eigener Serial-Port, eigene Datei ----

def microk_worker(stop_event):
    try:
        ser = serial.Serial(microk_port, 9600, serial.EIGHTBITS,
                             serial.PARITY_NONE, serial.STOPBITS_ONE, timeout=2)
    except Exception as e:
        print("[MicroK] Port-FEHLER:", e)
        return
    print("[MicroK] Port offen :", microk_port)
    # je SPRT-Kanal ein MEAS-Kommando (Ratio SPRT : Referenz)
    queries = [(ch, f"MEAS:RAT{ch}:REF{ref_ch}? 100,0.56 \r\n") for ch in sprt_chs]
    if not queries:
        print("[MicroK] Keine SPRT-Kanaele in param_combined.txt -> nichts zu messen.")
        ser.close()
        return
    start = time.time()
    n = 0
    print("[MicroK] Datei      :", microk_file)
    print("[MicroK] Warte auf Daten ...")
    try:
        with open(microk_file, "a") as f:
            while not stop_event.is_set():
                for ch, cmd in queries:
                    ser.write(cmd.encode("ascii"))
                    data = ""
                    while not data and not stop_event.is_set():
                        data = ser.readline().decode("utf-8", "replace").strip()
                    if not data:
                        continue
                    n += 1
                    t_min = (time.time() - start) / 60
                    f.write(f"{t_min};{datetime.now()};{data};0.56mA;Channel{ch};{n}\n")
                    f.flush()
                    print(f"[MicroK] #{n} Ch{ch}: {data}")
    finally:
        ser.close()
        print("[MicroK] gestoppt, Port geschlossen.")


def logger_worker(stop_event):
    try:
        ser = serial.Serial(logger_port, 19200, serial.EIGHTBITS,
                             serial.PARITY_NONE, serial.STOPBITS_ONE, timeout=1)
    except Exception as e:
        print("[TempLogger] Port-FEHLER:", e)
        return
    print("[TempLogger] Port offen :", logger_port)
    try:
        # --- Arduino/Kopf aufwecken und konfigurieren ---
        print("[TempLogger] Wecke Kopf, warte auf Antwort ...")
        ser.write("help\r\n".encode("ascii"))
        received = ""
        while not any(c.isalpha() for c in received) and not stop_event.is_set():
            received = ser.readline().decode("utf-8", "replace")
        print("[TempLogger] Kopf ist wach.")
        time.sleep(3)
        ser.write("LIVE \r\n".encode("ascii")); time.sleep(3)
        ser.write("nodes on \r\n".encode("ascii")); time.sleep(2); time.sleep(3)
        for command in on_commands + off_commands:
            print("[TempLogger] sende:", command.strip())
            ser.write(command.encode("ascii")); time.sleep(5)

        start = datetime.now()
        print("[TempLogger] Datei      :", logger_file)
        print("[TempLogger] Warte auf Messdaten ...")
        with open(logger_file, "a") as fh:
            while not stop_event.is_set():
                for g_idx in range(len(headers)):
                    ser.write(commands_groupNTCs[g_idx].encode("ascii"))
                    r = False
                    i = 0
                    while i < (Nr_MeasPoints + 3) and not stop_event.is_set():
                        received = ""
                        while not received and not stop_event.is_set():
                            received = ser.readline().decode("utf-8", "replace")
                        if not received:
                            continue
                        data_values = received.strip()
                        now = datetime.now()
                        sec = (now - start).total_seconds()
                        if "New Node Array:" in received:      # neue Node-Gruppe bestätigt
                            r = True
                            fh.write(f"Group{g_idx+1}; {headers[g_idx]}\r\n")
                        if r:
                            fh.write(f"Group{g_idx+1}; {sec}; {now}; {data_values}\n")
                            fh.flush()
                            i += 1
                            print(f"[TempLogger] G{g_idx+1} {i}/{Nr_MeasPoints + 3}: {data_values}")
    finally:
        ser.close()
        print("[TempLogger] gestoppt, Port geschlossen.")

In [ ]:
# ---- Meta-Datei mit allen Messparametern schreiben ----
meta_file = f"Output/{exp}_{run_stamp}_meta.txt"
with open(meta_file, "w") as m:
    m.write(f"Experiment       : {exp}\n")
    m.write(f"Start (PC-Zeit)  : {datetime.now()}\n")
    m.write(f"Run-Stempel      : {run_stamp}\n")
    m.write("\n--- Aufgeloeste Einstellungen ---\n")
    m.write(f"MicroK-Port      : {microk_port}\n")
    m.write(f"MicroK-Kanaele   : Referenz={ref_ch}  SPRT={sprt_chs}\n")
    m.write(f"TempLogger-Port  : {logger_port}\n")
    m.write(f"NTC-Readout      : {active}\n")
    m.write(f"Gruppen          : {Nr_NTCs_group} Sensoren/Gruppe, {Nr_MeasPoints} Messpunkte\n")
    m.write(f"Node-IDs         : {Logger_sensorNo}\n")
    m.write(f"MicroK-Datei     : {microk_file}\n")
    m.write(f"NTC-Datei        : {logger_file}\n")
    m.write("\n--- Woertliche Kopie von param_combined.txt ---\n")
    with open(os.path.join(script_dir, "param_combined.txt")) as p:
        m.write(p.read())
print("Meta-Datei geschrieben:", meta_file)


# ---- Beide Threads starten und bis zum Interrupt laufen lassen ----
stop_event = threading.Event()
t_micro  = threading.Thread(target=microk_worker, args=(stop_event,), daemon=True, name="MicroK")
t_logger = threading.Thread(target=logger_worker, args=(stop_event,), daemon=True, name="Logger")

t_micro.start()
t_logger.start()
print("Beide Logger laufen parallel. Stoppen: Kernel unterbrechen (Interrupt / Ctrl-C).")

try:
    while t_micro.is_alive() or t_logger.is_alive():
        time.sleep(0.5)
except KeyboardInterrupt:
    print("\nStoppe beide ...")
    stop_event.set()
    t_micro.join(timeout=15)
    t_logger.join(timeout=15)
    print("Fertig. Dateien liegen in ./Output/")